In [21]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [23]:
# ── Style
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f8f8",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "font.size": 11,
})
 
PURPLE = "#534AB7"
CORAL  = "#D85A30"
TEAL   = "#1D9E75"
GRAY   = "#888780"
AMBER  = "#BA7517"

In [24]:
def black_scholes(S, K, r, sigma, T, option="call"):
    """Analytical Black-Scholes price for a European option."""
    if T <= 0:
        return max(S - K, 0) if option == "call" else max(K - S, 0)
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option == "call":
        return S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    else:
        return K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)
 
 
def monte_carlo_price(S, K, r, sigma, T, N=200_000, option="call", seed=42):
    """
    Price a European option via Monte Carlo simulation.
 
    Under risk-neutral measure, the stock at expiry follows:
        S_T = S_0 * exp((r - 0.5σ²)T + σ√T · Z)
    where Z ~ N(0,1).
 
    Returns (price, std_error, array_of_payoffs)
    """
    rng = np.random.default_rng(seed)
    Z   = rng.standard_normal(N)
    ST  = S * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
 
    if option == "call":
        payoffs = np.maximum(ST - K, 0)
    else:
        payoffs = np.maximum(K - ST, 0)
 
    discount  = np.exp(-r * T)
    price     = discount * payoffs.mean()
    std_error = discount * payoffs.std() / np.sqrt(N)
    return price, std_error, payoffs
 
 
def simulate_gbm_paths(S0, r, sigma, T, n_steps=252, n_paths=50, seed=0):
    """
    Simulate n_paths stock price paths using Geometric Brownian Motion.
 
    GBM: dS = r·S·dt + σ·S·dW
    Exact discretisation: S_{t+dt} = S_t · exp((r - 0.5σ²)dt + σ√dt · Z)
    """
    rng  = np.random.default_rng(seed)
    dt   = T / n_steps
    t    = np.linspace(0, T, n_steps + 1)
    Z    = rng.standard_normal((n_paths, n_steps))
    log_returns = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_paths   = np.concatenate(
        [np.zeros((n_paths, 1)), np.cumsum(log_returns, axis=1)], axis=1
    )
    paths = S0 * np.exp(log_paths)
    return t, paths

def compute_greeks(S, K, r, sigma, T, N=300_000, bump_pct=0.01):
    h_s     = S * bump_pct
    h_sigma = sigma * bump_pct
    h_T     = T * bump_pct
    h_r     = r * bump_pct if r > 0 else 0.001

    mc = lambda **kw: monte_carlo_price(**{"S":S,"K":K,"r":r,"sigma":sigma,"T":T,"N":N, **kw})[0]

    c0   = mc()
    c_su = mc(S=S + h_s)
    c_sd = mc(S=S - h_s)
    c_vu = mc(sigma=sigma + h_sigma)
    c_vd = mc(sigma=sigma - h_sigma)
    c_Td = mc(T=T - h_T)
    c_ru = mc(r=r + h_r)
    c_rd = mc(r=r - h_r)

    delta = (c_su - c_sd) / (2 * h_s)
    gamma = (c_su - 2 * c0 + c_sd) / (h_s**2)
    vega  = (c_vu - c_vd) / (2 * h_sigma)
    theta = (c_Td - c0) / h_T
    rho   = (c_ru - c_rd) / (2 * h_r)

    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    bs_delta = norm.cdf(d1)
    bs_gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    bs_vega  = S * norm.pdf(d1) * np.sqrt(T)
    bs_theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
                - r * K * np.exp(-r*T) * norm.cdf(d2))
    bs_rho   = K * T * np.exp(-r*T) * norm.cdf(d2)

    return {
        "greek": ["Delta", "Gamma", "Vega", "Theta (per yr)", "Rho"],
        "mc":    [delta, gamma, vega, theta, rho],
        "bs":    [bs_delta, bs_gamma, bs_vega, bs_theta, bs_rho],
    }
def convergence_analysis(S, K, r, sigma, T, ns=None):
    """
    Run MC pricer at increasing N and record price ± 95% CI.
    Returns (ns, prices, lower_ci, upper_ci, bs_price).
    """
    if ns is None:
        ns = np.unique(np.logspace(2, 6, 60).astype(int))
    bs = black_scholes(S, K, r, sigma, T, "call")
    prices, lo, hi = [], [], []
    rng = np.random.default_rng(99)
    for n in ns:
        Z  = rng.standard_normal(n)
        ST = S * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
        p  = np.maximum(ST - K, 0)
        disc = np.exp(-r*T)
        mu   = disc * p.mean()
        se   = disc * p.std() / np.sqrt(n)
        prices.append(mu)
        lo.append(mu - 1.96*se)
        hi.append(mu + 1.96*se)
    return ns, np.array(prices), np.array(lo), np.array(hi), bs
 

In [27]:
def plot_paths(S0, K, r, sigma, T):
    t, paths = simulate_gbm_paths(S0, r, sigma, T, n_steps=252, n_paths=60)
 
    fig, ax = plt.subplots(figsize=(10, 5))
    final_prices = paths[:, -1]
 
    for i, path in enumerate(paths):
        color = TEAL if path[-1] >= K else CORAL
        ax.plot(t, path, color=color, alpha=0.35, linewidth=0.8)
 
    ax.axhline(K, color=AMBER, linewidth=1.8, linestyle="--", label=f"Strike K = ${K}")
    ax.axhline(S0, color=GRAY, linewidth=1.2, linestyle=":", label=f"S₀ = ${S0}")
 
    itm = (final_prices >= K).sum()
    ax.set_xlabel("Time (years)")
    ax.set_ylabel("Stock price ($)")
    ax.set_title(
        f"GBM simulated paths  |  σ={sigma:.0%}  r={r:.0%}  T={T}yr\n"
        f"{itm}/{len(paths)} paths expire in-the-money  ({TEAL} = ITM, {CORAL} doesn't apply — green=ITM, red=OTM)",
        fontsize=10,
    )
    ax.legend(fontsize=10)
    # add mini payoff note
    ax.text(T * 1.01, K + 2, "Strike", color=AMBER, fontsize=9, va="bottom")
    plt.tight_layout()
    plt.savefig("C:/Users/Shivom Srivastava/OneDrive/Desktop/DataProjects/1_gbm_paths.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Saved: 1_gbm_paths.png")
 
 
def plot_greeks(S, K, r, sigma, T):
    g = compute_greeks(S, K, r, sigma, T)
    names = g["greek"]
    mc_vals = g["mc"]
    bs_vals = g["bs"]
 
    x = np.arange(len(names))
    width = 0.35
 
    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, mc_vals, width, label="Monte Carlo", color=PURPLE, alpha=0.85)
    bars2 = ax.bar(x + width/2, bs_vals,  width, label="Black-Scholes", color=TEAL,   alpha=0.85)
 
    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylabel("Greek value")
    ax.set_title("Greeks: Monte Carlo (finite difference) vs Black-Scholes", fontsize=12)
    ax.legend()
    ax.axhline(0, color=GRAY, linewidth=0.8)
 
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002 if h >= 0 else h - 0.015,
                f"{h:.3f}", ha="center", va="bottom" if h >= 0 else "top", fontsize=8.5, color=PURPLE)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002 if h >= 0 else h - 0.015,
                f"{h:.3f}", ha="center", va="bottom" if h >= 0 else "top", fontsize=8.5, color=TEAL)
 
    plt.tight_layout()
    plt.savefig("C:/Users/Shivom Srivastava/OneDrive/Desktop/DataProjects/2_greeks.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Saved: 2_greeks.png")
 
 
def plot_put_call_parity(S, K, r, sigma, T, N=200_000):
    call_mc, call_se, _ = monte_carlo_price(S, K, r, sigma, T, N, "call")
    put_mc,  put_se,  _ = monte_carlo_price(S, K, r, sigma, T, N, "put",  seed=1)
 
    # Put-call parity: C - P = S - K·e^{-rT}
    lhs           = call_mc - put_mc
    rhs           = S - K * np.exp(-r * T)
    call_bs       = black_scholes(S, K, r, sigma, T, "call")
    put_bs        = black_scholes(S, K, r, sigma, T, "put")
    lhs_bs        = call_bs - put_bs
 
    # Sweep stock prices to show parity holds across moneyness
    S_range = np.linspace(60, 160, 60)
    lhs_sweep, rhs_sweep = [], []
    for s in S_range:
        c, _, _ = monte_carlo_price(s, K, r, sigma, T, 80_000, "call", seed=int(s))
        p, _, _ = monte_carlo_price(s, K, r, sigma, T, 80_000, "put",  seed=int(s)+1)
        lhs_sweep.append(c - p)
        rhs_sweep.append(s - K * np.exp(-r * T))
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
    # Left: bar comparison at base params
    ax = axes[0]
    labels = ["C − P\n(MC)", "S − Ke^{-rT}\n(theory)", "C − P\n(BS)"]
    vals   = [lhs, rhs, lhs_bs]
    colors = [PURPLE, TEAL, CORAL]
    bars   = ax.bar(labels, vals, color=colors, alpha=0.85, width=0.4)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.1,
                f"${v:.3f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_ylabel("Value ($)")
    ax.set_title(f"Put-Call Parity check\nS={S}  K={K}  r={r:.0%}  σ={sigma:.0%}  T={T}yr", fontsize=10)
    ax.set_ylim(min(vals)*1.3 - 2, max(vals)*1.3 + 2)
 
    # Right: parity across moneyness
    ax2 = axes[1]
    ax2.plot(S_range, lhs_sweep, color=PURPLE, label="C − P  (MC)", linewidth=1.8)
    ax2.plot(S_range, rhs_sweep, color=TEAL,   label="S − Ke^{-rT}", linewidth=1.8, linestyle="--")
    ax2.axvline(K, color=AMBER, linestyle=":", linewidth=1.2, label=f"ATM  K={K}")
    ax2.set_xlabel("Stock price S")
    ax2.set_ylabel("Value ($)")
    ax2.set_title("Parity holds across all moneyness levels", fontsize=10)
    ax2.legend(fontsize=9)
 
    plt.tight_layout()
    plt.savefig("C:/Users/Shivom Srivastava/OneDrive/Desktop/DataProjects/3_put_call_parity.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Saved: 3_put_call_parity.png")
 
 
def plot_convergence(S, K, r, sigma, T):
    ns, prices, lo, hi, bs = convergence_analysis(S, K, r, sigma, T)
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
 
    ax = axes[0]
    ax.semilogx(ns, prices, color=PURPLE, linewidth=1.8, label="MC price")
    ax.fill_between(ns, lo, hi, color=PURPLE, alpha=0.18, label="95% CI")
    ax.axhline(bs, color=CORAL, linewidth=1.8, linestyle="--", label=f"BS = ${bs:.4f}")
    ax.set_xlabel("Number of simulations (log scale)")
    ax.set_ylabel("Option price ($)")
    ax.set_title("MC price convergence to Black-Scholes", fontsize=11)
    ax.legend(fontsize=9)
 
    # Right: std error vs N
    ax2 = axes[1]
    std_errors = (hi - lo) / (2 * 1.96)
    theory_se  = np.exp(-r*T) * prices.std() / np.sqrt(ns)   # rough theoretical SE ∝ 1/√N
    ax2.loglog(ns, std_errors, color=TEAL, linewidth=1.8, label="Empirical std error")
    # overlay 1/√N reference line
    ref = std_errors[0] * np.sqrt(ns[0]) / np.sqrt(ns)
    ax2.loglog(ns, ref, color=GRAY, linewidth=1.2, linestyle=":", label="1/√N reference")
    ax2.set_xlabel("Number of simulations (log scale)")
    ax2.set_ylabel("Std error ($)")
    ax2.set_title("Standard error decays as 1/√N", fontsize=11)
    ax2.legend(fontsize=9)
 
    plt.tight_layout()
    plt.savefig("C:/Users/Shivom Srivastava/OneDrive/Desktop/DataProjects/4_convergence.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("✓ Saved: 4_convergence.png")

In [28]:
def main():
    # ── Parameters ──────────────────────────────────────────────────────
    S0    = 100      # current stock price
    K     = 105      # strike price (slightly OTM call)
    r     = 0.05     # annual risk-free rate
    sigma = 0.20     # annual volatility
    T     = 1.0      # time to expiry (years)
    N     = 200_000  # simulations for the base pricer
 
    print("=" * 55)
    print("  Monte Carlo Option Pricing — Full Project")
    print("=" * 55)
    print(f"  S={S0}  K={K}  r={r:.0%}  σ={sigma:.0%}  T={T}yr  N={N:,}")
    print()
 
    # ── 1. Base pricer ───────────────────────────────────────────────────
    call_mc, call_se, payoffs = monte_carlo_price(S0, K, r, sigma, T, N, "call")
    call_bs = black_scholes(S0, K, r, sigma, T, "call")
    print(f"[1] Base pricer")
    print(f"    MC call price : ${call_mc:.4f}  ±{call_se:.4f}  (95% CI: ${call_mc-1.96*call_se:.4f} – ${call_mc+1.96*call_se:.4f})")
    print(f"    BS call price : ${call_bs:.4f}")
    print(f"    Difference    : ${abs(call_mc - call_bs):.4f}")
    print()
 
    # ── 2. GBM paths ────────────────────────────────────────────────────
    print("[2] Plotting GBM paths …")
    plot_paths(S0, K, r, sigma, T)
    print()
 
    # ── 3. Greeks ───────────────────────────────────────────────────────
    print("[3] Computing Greeks via finite difference …")
    g = compute_greeks(S0, K, r, sigma, T)
    header = f"    {'Greek':<18} {'MC':>10}  {'BS':>10}  {'Diff':>10}"
    print(header)
    print("    " + "-" * (len(header) - 4))
    for name, mc_v, bs_v in zip(g["greek"], g["mc"], g["bs"]):
        print(f"    {name:<18} {mc_v:>10.4f}  {bs_v:>10.4f}  {abs(mc_v-bs_v):>10.4f}")
    plot_greeks(S0, K, r, sigma, T)
    print()
 
    # ── 4. Put-Call Parity ───────────────────────────────────────────────
    print("[4] Verifying Put-Call Parity: C - P = S - K·e^{-rT}")
    call_mc2, _, _ = monte_carlo_price(S0, K, r, sigma, T, N, "call", seed=10)
    put_mc2,  _, _ = monte_carlo_price(S0, K, r, sigma, T, N, "put",  seed=11)
    lhs = call_mc2 - put_mc2
    rhs = S0 - K * np.exp(-r * T)
    print(f"    C - P  (MC)      = ${lhs:.4f}")
    print(f"    S - Ke^{{-rT}}     = ${rhs:.4f}")
    print(f"    Parity error     = ${abs(lhs - rhs):.4f}")
    plot_put_call_parity(S0, K, r, sigma, T, N)
    print()
 
    # ── 5. Convergence ───────────────────────────────────────────────────
    print("[5] Running convergence analysis (100 → 1 000 000 sims) …")
    plot_convergence(S0, K, r, sigma, T)
    print()
 
    print("=" * 55)
    print("  All done! 4 plots saved to outputs/")
    print("=" * 55)
 
main()

  Monte Carlo Option Pricing — Full Project
  S=100  K=105  r=5%  σ=20%  T=1.0yr  N=200,000

[1] Base pricer
    MC call price : $8.0398  ±0.0297  (95% CI: $7.9815 – $8.0980)
    BS call price : $8.0214
    Difference    : $0.0184

[2] Plotting GBM paths …
✓ Saved: 1_gbm_paths.png

[3] Computing Greeks via finite difference …
    Greek                      MC          BS        Diff
    -----------------------------------------------------
    Delta                  0.5416      0.5422      0.0006
    Gamma                  0.0199      0.0198      0.0001
    Vega                  39.7508     39.6705      0.0802
    Theta (per yr)        -6.2911     -6.2771      0.0140
    Rho                   46.1380     46.2015      0.0634
✓ Saved: 2_greeks.png

[4] Verifying Put-Call Parity: C - P = S - K·e^{-rT}
    C - P  (MC)      = $0.1258
    S - Ke^{-rT}     = $0.1209
    Parity error     = $0.0049
✓ Saved: 3_put_call_parity.png

[5] Running convergence analysis (100 → 1 000 000 sims) …
✓ Saved